# 02 Classical ML + CNN Intrusion Detection (NSL-KDD)

This notebook trains and compares classical machine learning models and three CNN-based deep learning models on `data/processed/nsl_kdd_processed.csv` for binary intrusion detection (`attack_binary`).

Workflow summary:

1. Load dataset and split `X` / `y`
2. Preprocess using `src/preprocessing.py`
3. Train and evaluate classical ML baselines
4. Prepare data for CNN
5. Train and evaluate three CNN architectures
6. Build final ranked comparison and save models
7. Add experimental observations

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

np.random.seed(42)

cwd = Path.cwd().resolve()
if (cwd / "src").exists():
    project_root = cwd
elif (cwd.parent / "src").exists():
    project_root = cwd.parent
else:
    raise RuntimeError("Could not locate project root containing 'src' directory.")

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

print(f"Project root: {project_root}")

Project root: C:\Users\gudal\OneDrive\Desktop\ShockAwareNeuroSymbolic


## 1. Load Dataset

Load the processed NSL-KDD dataset from `data/processed/nsl_kdd_processed.csv`, then separate:

- `X` = all feature columns
- `y` = `attack_binary`

The target column is dropped from features.

In [ ]:
from src.preprocessing import build_preprocessing_pipeline, train_test_split_data

data_path = project_root / "data" / "processed" / "nsl_kdd_processed.csv"
df = pd.read_csv(data_path)

if "attack_binary" not in df.columns:
    raise ValueError("Expected target column 'attack_binary' in the dataset.")

X = df.drop(columns=["attack_binary"]).copy()
y = df["attack_binary"].astype(int).copy()

print(f"Dataset path: {data_path}")
print(f"Dataset shape: {df.shape}")
print(f"Feature matrix shape: {X.shape}")
print(f"Target distribution:\n{y.value_counts(normalize=True).rename('ratio')}")

[preprocessing] number of categorical features: 4
[preprocessing] number of numeric features: 38
Dataset path: C:\Users\gudal\OneDrive\Desktop\ShockAwareNeuroSymbolic\data\processed\nsl_kdd_processed.csv
Dataset shape: (148517, 43)
Train shape: (118813, 42) | Test shape: (29704, 42)
Encoded train shape: (118813, 160) | Encoded test shape: (29704, 160)
Train attack ratio: 0.4812 | Test attack ratio: 0.4812


## 2. Preprocess Features

Use the existing preprocessing pipeline from `src/preprocessing.py` and perform train/test split with:

- `test_size=0.2`
- `stratify=y`
- `random_state=42`

Categorical features of interest in NSL-KDD are: `protocol_type`, `service`, `flag`.
The pipeline applies one-hot encoding for categorical features and standard scaling for numeric features.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split_data(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
 )

preprocessor = build_preprocessing_pipeline(X_train)
X_train_pre = preprocessor.fit_transform(X_train)
X_test_pre = preprocessor.transform(X_test)

expected_categorical = {"protocol_type", "service", "flag"}
present_categorical = set(X_train.select_dtypes(include=["object", "category", "bool"]).columns)
missing_expected = expected_categorical - set(X_train.columns)

print(f"Train shape: {X_train.shape} | Test shape: {X_test.shape}")
print(f"Encoded train shape: {X_train_pre.shape} | Encoded test shape: {X_test_pre.shape}")
print(f"Train attack ratio: {y_train.mean():.4f} | Test attack ratio: {y_test.mean():.4f}")
print(f"Categorical features detected: {sorted(present_categorical)}")

if missing_expected:
    print(f"Warning: expected NSL-KDD categorical columns missing from data: {sorted(missing_expected)}")
else:
    print("Expected NSL-KDD categorical columns are present.")

## 3. Train Baseline Models

Train the required classical models from `src/baseline_models.py`:

- Logistic Regression
- Decision Tree
- Random Forest
- Extra Trees
- Gradient Boosting
- XGBoost
- LightGBM
- SVM (linear)
- SVM (RBF)
- MLP Classifier
- Balanced Random Forest

Evaluate with: Accuracy, Precision, Recall, F1 Score, ROC-AUC, PR-AUC.

In [ ]:
import matplotlib.pyplot as plt
from scipy import sparse

from src.baseline_models import train_all_models
from src.evaluation import evaluate_model

models_dir = project_root / "models"
models_dir.mkdir(parents=True, exist_ok=True)

requested_baseline_models = [
    "logistic_regression",
    "decision_tree",
    "random_forest",
    "extra_trees",
    "gradient_boosting",
    "xgboost",
    "lightgbm",
    "svm_linear",
    "svm_rbf",
    "mlp_classifier",
    "balanced_random_forest",
]

display_names = {
    "logistic_regression": "Logistic Regression",
    "decision_tree": "Decision Tree",
    "random_forest": "Random Forest",
    "extra_trees": "Extra Trees",
    "gradient_boosting": "Gradient Boosting",
    "xgboost": "XGBoost",
    "lightgbm": "LightGBM",
    "svm_linear": "SVM (linear)",
    "svm_rbf": "SVM (RBF)",
    "mlp_classifier": "MLP Classifier",
    "balanced_random_forest": "Balanced Random Forest",
}

dense_needed = {"gradient_boosting", "mlp_classifier"}

all_trained_models = train_all_models(X_train_pre, y_train, random_state=42)
trained_models = {
    model_name: all_trained_models[model_name]
    for model_name in requested_baseline_models
    if model_name in all_trained_models
}

missing_models = [name for name in requested_baseline_models if name not in trained_models]
if missing_models:
    raise RuntimeError(
        "Required baseline models are unavailable. Install missing dependencies and rerun: "
        + ", ".join(missing_models)
    )

baseline_rows = []
for model_name, model in trained_models.items():
    X_eval = X_test_pre.toarray() if sparse.issparse(X_test_pre) and model_name in dense_needed else X_test_pre
    metrics = evaluate_model(model, X_eval, y_test)

    cm_fig = metrics.pop("confusion_matrix_figure")
    roc_fig = metrics.pop("roc_curve_figure")
    plt.close(cm_fig)
    plt.close(roc_fig)

    baseline_rows.append(
        {
            "model": display_names.get(model_name, model_name),
            "model_key": model_name,
            "model_family": "classical_ml",
            "accuracy": metrics["accuracy"],
            "precision": metrics["precision"],
            "recall": metrics["recall"],
            "f1": metrics["f1_score"],
            "roc_auc": metrics["roc_auc"],
            "pr_auc": metrics["pr_auc"],
        }
    )

    joblib.dump(model, models_dir / f"{model_name}.joblib")

joblib.dump(preprocessor, models_dir / "preprocessing_pipeline.joblib")

baseline_results_df = pd.DataFrame(baseline_rows).sort_values("roc_auc", ascending=False).reset_index(drop=True)
display(Markdown("### Classical ML Results"))
display(baseline_results_df[["model", "accuracy", "precision", "recall", "f1", "roc_auc", "pr_auc"]])

print(f"Saved classical models to: {models_dir}")
print("Saved preprocessing pipeline to models/preprocessing_pipeline.joblib")

,model,accuracy,precision,recall,f1_score,roc_auc,pr_auc,brier_score
0,balanced_random_forest,0.766324,0.966572,0.610613,0.748424,0.966258,0.970548,0.154875
1,random_forest,0.764727,0.966654,0.607652,0.746220,0.965317,0.970197,0.154777
2,lightgbm,0.777945,0.967730,0.630951,0.763868,0.960475,0.963992,0.213299
3,xgboost_weighted,0.786107,0.967878,0.645679,0.774610,0.960207,0.961483,0.188478
4,xgboost,0.784643,0.968083,0.642874,0.772653,0.957248,0.957078,0.190482
5,extra_trees,0.775018,0.957660,0.632744,0.762012,0.953098,0.960155,0.158415
6,gradient_boosting,0.799414,0.971092,0.667498,0.791170,0.936602,0.948099,0.176325
7,mlp_classifier,0.795289,0.927042,0.695083,0.794478,0.896109,0.927436,0.199875
8,svm_rbf,0.773510,0.960983,0.627601,0.759310,0.883677,0.919521,0.204213
9,elasticnet_logistic,0.749601,0.915491,0.617081,0.737234,0.829093,0.878813,0.223584


Saved 14 trained models to: C:\Users\gudal\OneDrive\Desktop\ShockAwareNeuroSymbolic\models
Saved preprocessing pipeline to models/preprocessing_pipeline.joblib


## 4-7. Prepare Data for CNN and Define Three CNN Architectures

Convert preprocessed features to NumPy arrays and reshape for 1D CNN input:

- `X_train_cnn = X_train.reshape(samples, features, 1)`
- `X_test_cnn = X_test.reshape(samples, features, 1)`

Number of classes is binary (`2`).

CNN architectures:

1. CNN Model 1 (Basic): `Conv1D(32) -> MaxPool -> Flatten -> Dense(64) -> Dense(1,sigmoid)`
2. CNN Model 2 (Deep): `Conv1D(32) -> ReLU -> Conv1D(64) -> ReLU -> MaxPool -> Dropout(0.3) -> Flatten -> Dense(128) -> Dense(1,sigmoid)`
3. CNN Model 3 (Advanced): `Conv1D(32) -> BatchNorm -> Conv1D(64) -> ReLU -> MaxPool -> Dropout(0.4) -> Flatten -> Dense(128) -> Dense(64) -> Dense(1,sigmoid)`

In [ ]:
from sklearn.metrics import accuracy_score, average_precision_score, f1_score, precision_score, recall_score, roc_auc_score

try:
    import tensorflow as tf
except Exception as exc:
    raise RuntimeError(
        "TensorFlow is required for CNN training. Use an environment/kernel with TensorFlow installed."
    ) from exc

X_train_dense = X_train_pre.toarray() if hasattr(X_train_pre, "toarray") else np.asarray(X_train_pre)
X_test_dense = X_test_pre.toarray() if hasattr(X_test_pre, "toarray") else np.asarray(X_test_pre)

y_train_cnn = y_train.astype(int).to_numpy()
y_test_cnn = y_test.astype(int).to_numpy()

feature_count = int(X_train_dense.shape[1])
X_train_cnn = X_train_dense.astype(np.float32).reshape((X_train_dense.shape[0], feature_count, 1))
X_test_cnn = X_test_dense.astype(np.float32).reshape((X_test_dense.shape[0], feature_count, 1))

num_classes = 2

def build_cnn_model_1() -> tf.keras.Model:
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(feature_count, 1)),
        tf.keras.layers.Conv1D(32, kernel_size=3, activation="relu"),
        tf.keras.layers.MaxPooling1D(pool_size=2),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(64, activation="relu"),
        tf.keras.layers.Dense(1, activation="sigmoid"),
    ])
    return model

def build_cnn_model_2() -> tf.keras.Model:
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(feature_count, 1)),
        tf.keras.layers.Conv1D(32, kernel_size=3),
        tf.keras.layers.ReLU(),
        tf.keras.layers.Conv1D(64, kernel_size=3),
        tf.keras.layers.ReLU(),
        tf.keras.layers.MaxPooling1D(pool_size=2),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(128, activation="relu"),
        tf.keras.layers.Dense(1, activation="sigmoid"),
    ])
    return model

def build_cnn_model_3() -> tf.keras.Model:
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(feature_count, 1)),
        tf.keras.layers.Conv1D(32, kernel_size=3),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Conv1D(64, kernel_size=3),
        tf.keras.layers.ReLU(),
        tf.keras.layers.MaxPooling1D(pool_size=2),
        tf.keras.layers.Dropout(0.4),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(128, activation="relu"),
        tf.keras.layers.Dense(64, activation="relu"),
        tf.keras.layers.Dense(1, activation="sigmoid"),
    ])
    return model

cnn_builders = {
    "cnn_model_1_basic": build_cnn_model_1,
    "cnn_model_2_deep": build_cnn_model_2,
    "cnn_model_3_advanced": build_cnn_model_3,
}

print(f"CNN train shape: {X_train_cnn.shape}")
print(f"CNN test shape: {X_test_cnn.shape}")
print(f"Number of classes: {num_classes}")

Deep-learning backend: torch
CNN input shape: (channels=1, length=118)
CNN classes: 2
Device: cpu
Implemented CNN models:
- cnn_model_1_basic
- cnn_model_2_deep
- cnn_model_3_advanced


In [ ]:
tf.keras.utils.set_random_seed(42)

cnn_models_dir = models_dir / "cnn"
cnn_models_dir.mkdir(parents=True, exist_ok=True)

cnn_display_names = {
    "cnn_model_1_basic": "CNN Model 1 (Basic)",
    "cnn_model_2_deep": "CNN Model 2 (Deep)",
    "cnn_model_3_advanced": "CNN Model 3 (Advanced)",
}

cnn_rows = []
for cnn_key, build_fn in cnn_builders.items():
    model = build_fn()
    model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"] )

    model.fit(
        X_train_cnn,
        y_train_cnn,
        epochs=10,
        batch_size=128,
        verbose=0,
    )

    y_proba = model.predict(X_test_cnn, verbose=0).ravel()
    y_pred = (y_proba >= 0.5).astype(int)

    cnn_rows.append(
        {
            "model": cnn_display_names[cnn_key],
            "model_key": cnn_key,
            "model_family": "cnn",
            "accuracy": float(accuracy_score(y_test_cnn, y_pred)),
            "precision": float(precision_score(y_test_cnn, y_pred, zero_division=0)),
            "recall": float(recall_score(y_test_cnn, y_pred, zero_division=0)),
            "f1": float(f1_score(y_test_cnn, y_pred, zero_division=0)),
            "roc_auc": float(roc_auc_score(y_test_cnn, y_proba)),
            "pr_auc": float(average_precision_score(y_test_cnn, y_proba)),
        }
    )

    model.save(cnn_models_dir / f"{cnn_key}.keras")

cnn_results_df = pd.DataFrame(cnn_rows).sort_values("roc_auc", ascending=False).reset_index(drop=True)
display(Markdown("### CNN Results"))
display(cnn_results_df[["model", "accuracy", "precision", "recall", "f1", "roc_auc", "pr_auc"]])

model_comparison_df = (
    pd.concat([baseline_results_df, cnn_results_df], ignore_index=True)
    .sort_values("roc_auc", ascending=False)
    .reset_index(drop=True)
)

display(Markdown("### Final Model Comparison (Ranked by ROC-AUC)"))
display(model_comparison_df[["model", "model_family", "accuracy", "precision", "recall", "f1", "roc_auc", "pr_auc"]])

print(f"Saved CNN models to: {cnn_models_dir}")

,model,backend,epochs_trained,accuracy,precision,recall,f1_score,roc_auc,pr_auc,brier_score
0,cnn_model_2_deep,torch,12,0.797330,0.926243,0.699681,0.797177,0.930574,0.945149,0.186647
1,cnn_model_3_advanced,torch,12,0.786773,0.924746,0.680823,0.784256,0.913132,0.935716,0.189527
2,cnn_model_1_basic,torch,12,0.777901,0.924311,0.664225,0.772977,0.897021,0.924599,0.201529


Saved 3 CNN models to: C:\Users\gudal\OneDrive\Desktop\ShockAwareNeuroSymbolic\models\cnn


### Combined Baseline + CNN Ranking (by ROC-AUC)

,model,accuracy,precision,recall,f1_score,roc_auc,pr_auc,brier_score,model_family,backend,epochs_trained
0,balanced_random_forest,0.766324,0.966572,0.610613,0.748424,0.966258,0.970548,0.154875,baseline,NaN,NaN
1,random_forest,0.764727,0.966654,0.607652,0.746220,0.965317,0.970197,0.154777,baseline,NaN,NaN
2,lightgbm,0.777945,0.967730,0.630951,0.763868,0.960475,0.963992,0.213299,baseline,NaN,NaN
3,xgboost_weighted,0.786107,0.967878,0.645679,0.774610,0.960207,0.961483,0.188478,baseline,NaN,NaN
4,xgboost,0.784643,0.968083,0.642874,0.772653,0.957248,0.957078,0.190482,baseline,NaN,NaN
5,extra_trees,0.775018,0.957660,0.632744,0.762012,0.953098,0.960155,0.158415,baseline,NaN,NaN
6,gradient_boosting,0.799414,0.971092,0.667498,0.791170,0.936602,0.948099,0.176325,baseline,NaN,NaN
7,cnn_model_2_deep,0.797330,0.926243,0.699681,0.797177,0.930574,0.945149,0.186647,cnn,torch,12.0
8,cnn_model_3_advanced,0.786773,0.924746,0.680823,0.784256,0.913132,0.935716,0.189527,cnn,torch,12.0
9,cnn_model_1_basic,0.777901,0.924311,0.664225,0.772977,0.897021,0.924599,0.201529,cnn,torch,12.0


In [ ]:
display(Markdown("## 8-11. CNN Evaluation, Final Comparison, Model Saving, and Experimental Observations"))

best_overall = model_comparison_df.iloc[0]
best_classical = model_comparison_df[model_comparison_df["model_family"] == "classical_ml"].iloc[0]
best_cnn = model_comparison_df[model_comparison_df["model_family"] == "cnn"].iloc[0]

classical_mean_recall = model_comparison_df.loc[model_comparison_df["model_family"] == "classical_ml", "recall"].mean()
cnn_mean_recall = model_comparison_df.loc[model_comparison_df["model_family"] == "cnn", "recall"].mean()
classical_mean_pr_auc = model_comparison_df.loc[model_comparison_df["model_family"] == "classical_ml", "pr_auc"].mean()
cnn_mean_pr_auc = model_comparison_df.loc[model_comparison_df["model_family"] == "cnn", "pr_auc"].mean()

observations_md = f"""
### Experimental Observations

- Best overall model: **{best_overall['model']}** ({best_overall['model_family']}) with ROC-AUC **{best_overall['roc_auc']:.4f}** and F1 **{best_overall['f1']:.4f}**.
- Best classical model: **{best_classical['model']}** with ROC-AUC **{best_classical['roc_auc']:.4f}**.
- Best CNN model: **{best_cnn['model']}** with ROC-AUC **{best_cnn['roc_auc']:.4f}**.
- Classical ML average recall: **{classical_mean_recall:.4f}** vs CNN average recall: **{cnn_mean_recall:.4f}**.
- Classical ML average PR-AUC: **{classical_mean_pr_auc:.4f}** vs CNN average PR-AUC: **{cnn_mean_pr_auc:.4f}**.
- Intrusion detection focus: prioritize **recall** and **PR-AUC** to reduce missed attacks while maintaining acceptable precision.
"""

display(Markdown(observations_md))

print("Classical models saved under models/*.joblib")
print("CNN models saved under models/cnn/*.keras")

### CNN Execution Summary

- **cnn_model_2_deep** (torch): Accuracy=0.7973, Recall=0.6997, ROC-AUC=0.9306, PR-AUC=0.9451
- **cnn_model_3_advanced** (torch): Accuracy=0.7868, Recall=0.6808, ROC-AUC=0.9131, PR-AUC=0.9357
- **cnn_model_1_basic** (torch): Accuracy=0.7779, Recall=0.6642, ROC-AUC=0.8970, PR-AUC=0.9246

### Notes

- Run cells from top to bottom to reproduce splits, model training, and saved artifacts.
- Output files are stored in `models/` and `models/cnn/`.
- The final ranking table is sorted by ROC-AUC across classical ML and CNN models.